# 🧪 External-data benchmark (competition-compliant)

**Rules this notebook follows** (organisers' clarification): if you train, use only external data; every tuned parameter
is tuned on external data, never on the ground-truth possession data or the provided video. So:

* the **public ground truth is not read anywhere in this notebook**;
* models train on the seven **IDSSE** matches (Sportec/DFL, CC-BY 4.0; the provider's own `BallStatus` / `BallPossession`
  labels) and are validated **leave-one-match-out**: every match is predicted by a model that never saw it;
* decoder settings (smoothing width, minimum possession length, contact filter) are chosen on those external predictions;
  `tuned_on_others` below scores each match with settings picked from the *other* six, an honest transfer estimate;
* the provided match contributes only its tracking: as model input, and to copy where its ball goes unseen (so IDSSE's
  always-present ball has the same gaps).

Metrica's sample games are not used: their README states no licence (confirm with the organisers before adding them).
Caveat that matters: IDSSE's possession definition is more fragmented than the competition's, so absolute scores here are
not predictions of the leaderboard. Use them to compare model families and decoder rules.

Data: `python starter/external.py --fetch-idsse` (2.6 GB, git-ignored). Cite the IDSSE dataset (Bassek et al., figshare
doi 10.6084/m9.figshare.28196177).

In [1]:
import os, sys, time, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "starter" else os.getcwd()
sys.path.insert(0, os.path.join(ROOT, "starter")); sys.path.insert(0, os.path.join(ROOT, "evaluation"))
import possession_metrics as pm
import external as ext
import external_pipeline as ep
pd.set_option("display.width", 200)
if not all(os.path.isfile(p) for m in ext.IDSSE_MATCHES for p in ext.idsse_files(m).values()):
    ext.fetch_idsse()

## 1. Load the seven matches

In [2]:
sets = ep.load_idsse_sets(ep.ball_unseen_pattern())
def stats(m, d):
    y = d["y"]; sp = pm.possession_spells(y, pm.valid_runs(y)); L = np.array([(b - a) / 25 for a, b, _ in sp])
    return dict(match=m, frames=len(y), H=(y == 0).mean(), A=(y == 1).mean(), D=(y == 2).mean(),
                per_5min=len(sp) / (len(y) / 25 / 300), median_s=np.median(L), short_lt_4s=(L < 4).mean())
pd.DataFrame([stats(m, d) for m, d in sets.items()]).round(3)

,match,frames,H,A,D,per_5min,median_s,short_lt_4s
0,J03WMX,145967,0.268,0.317,0.415,26.256,3.00,0.573
1,J03WN1,141561,0.257,0.226,0.517,26.490,3.12,0.586
2,J03WOH,137214,0.285,0.239,0.476,21.973,3.94,0.502
3,J03WOY,142536,0.265,0.270,0.465,23.994,3.60,0.553
4,J03WPY,146211,0.350,0.207,0.443,21.596,3.84,0.525
5,J03WQQ,142345,0.260,0.332,0.407,21.128,4.68,0.454
6,J03WR9,146810,0.363,0.218,0.419,18.902,4.24,0.481


## 2. Leave-one-match-out posteriors for each model family

In [3]:
POSTS, t = {}, time.time()
for name in ep.MODELS:
    POSTS[name] = ep.lomo_posteriors(name, sets); print(name, round(time.time() - t), "s")

logreg 17 s


hierarchical 23 s


lightgbm 35 s


## 3. Tune the decoder on external predictions; compare models
Per held-out match: `plain_w25` = smoothing only; `tuned_on_others` = decoder settings picked on the other six matches'
predictions. Means and standard deviations are over the seven matches (a match is the unit of evidence here: roughly
300-500 provider-labelled possessions each).

In [4]:
TUNED, rows, details = {}, [], {}
for name in ep.MODELS:
    joint, table = ep.tune_decoder(POSTS[name], sets); TUNED[name] = joint; details[name] = pd.DataFrame(table)
    d = details[name]
    rows.append(dict(model=name, plain_w25=d.plain_w25.mean(), tuned_on_others=d.tuned_on_others.mean(),
                     sd_over_matches=d.tuned_on_others.std(), joint_decoder=str({k: v for k, v in joint.items() if k != "inflight_thr"})))
board = pd.DataFrame(rows).sort_values("tuned_on_others", ascending=False).reset_index(drop=True)
board.round(3)

,model,plain_w25,tuned_on_others,sd_over_matches,joint_decoder
0,lightgbm,0.432,0.472,0.035,"{'w': 25, 'min_s': 3, 'contact_m': 1.1}"
1,hierarchical,0.332,0.378,0.047,"{'w': 50, 'min_s': 3, 'contact_m': 1.1}"
2,logreg,0.312,0.353,0.044,"{'w': 25, 'min_s': 3, 'contact_m': 1.1}"


In [5]:
best = board.model.iloc[0]
print("selected on external data only:", best)
details[best].round(3)

selected on external data only: lightgbm


,match,tuned_on_others,plain_w25,params
0,J03WMX,0.439,0.403,"{'w': 25, 'min_s': 3, 'contact_m': 1.1}"
1,J03WN1,0.412,0.350,"{'w': 25, 'min_s': 2, 'contact_m': 1.1}"
2,J03WOH,0.488,0.466,"{'w': 25, 'min_s': 2, 'contact_m': 1.1}"
3,J03WOY,0.464,0.443,"{'w': 25, 'min_s': 3, 'contact_m': 1.1}"
4,J03WPY,0.489,0.449,"{'w': 25, 'min_s': 2, 'contact_m': 1.1}"
5,J03WQQ,0.504,0.459,"{'w': 25, 'min_s': 2, 'contact_m': 1.1}"
6,J03WR9,0.507,0.458,"{'w': 25, 'min_s': 3, 'contact_m': 1.1}"


## 4. Export the full-match submission
Train the selected family on all seven matches, apply the externally-tuned decoder to this match's two halves, write the
CSV, and validate its format. Then run the official scorer **once** from a terminal, to report the number
(`python evaluation/run_evaluation.py`); nothing above this line depends on it, and it must not be used to pick between
models, weights or settings.

In [6]:
import subprocess
from generate_full_submission import predict_and_write
path = predict_and_write(ep.fit(best, sets), TUNED[best], "ext_%s.csv" % best)
print(subprocess.run([sys.executable, os.path.join(ROOT, "evaluation", "validate_submission.py"), path],
                     capture_output=True, text=True).stdout[:900])

wrote /Users/mushroomyyy/more-than-a-hack-2027/evaluation/results/ext_lightgbm.csv


OK    ext_lightgbm.csv
      199319 rows, frame 0..199318

      label counts
        H    56394   28.3%
        A    35148   17.6%
        D    67315   33.8%
        X    40462   20.3%

      reports 331 possessions of mean 11.1 s
      (counted with each X as a gap; the scorer bridges abstentions inside a real
       possession, so its count can be lower than this one)

Valid. Note this checks the format only -- it says nothing about whether the labels are right.
Score it against the public ground truth with run_evaluation.py.

